# Lab Day 1: Neural Network

Notebook triển khai theo Part 0–4 trong GUIDE.md và RUBRIC.md. Split đã được tạo và được đóng gói cùng mã nguồn cho Google Colab. Notebook không tự chạy command terminal.

Trên Colab, ô đầu tiên yêu cầu mount Drive và upload gói v2 một lần, rồi tạo submission_2A202602777/ trong thư mục Track4-Day1-2A202602777-v2. Hãy mở notebook được sao chép trong submission_2A202602777/code/lab.ipynb để lưu output chạy vào đúng hồ sơ nộp. Không điền trước số liệu; nhận xét cần dựa trên kết quả bạn quan sát được.

In [ ]:
from pathlib import Path, PurePosixPath
import importlib, json, os, shutil, sys, zipfile
import numpy as np
import torch

STUDENT_ID = '2A202602777'
COLAB_ROOT = Path('/content/drive/MyDrive/Track4-Day1-2A202602777-v2')
try:
    from google.colab import drive, files
except ImportError:
    drive = files = None
if drive is not None:
    drive.mount('/content/drive')
    required = ('code/lab.ipynb', 'code/data.py', 'code/lab_workflow.py', 'scripts/split_data.py', 'data/processed/train.npz', 'data/processed/eval.npz')
    if not all((COLAB_ROOT / path).is_file() for path in required):
        print('Chọn file track4_day1_colab_2A202602777_v2.zip từ máy của bạn.')
        os.chdir('/content')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Chỉ upload một file ZIP của lab.')
        archive = Path('/content') / next(iter(uploaded))
        if archive.suffix.lower() != '.zip':
            raise RuntimeError('File upload phải là ZIP của lab.')
        COLAB_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as package:
            entries = [(item, item.filename.replace('\\', '/')) for item in package.infolist()]
            missing = set(required) - {name for _, name in entries}
            if missing:
                raise RuntimeError(f'ZIP thiếu: {sorted(missing)}')
            notebook_entry = next(item for item, name in entries if name == 'code/lab.ipynb')
            notebook = json.loads(package.read(notebook_entry))
            if 'Track4-Day1-2A202602777-v2' not in ''.join(notebook['cells'][1]['source']):
                raise RuntimeError('ZIP chứa notebook bản cũ; hãy tạo và upload ZIP v2.')
            for item, name in entries:
                path = PurePosixPath(name)
                if not path.parts or path.is_absolute() or '..' in path.parts or ':' in path.parts[0]:
                    raise RuntimeError(f'ZIP chứa đường dẫn không hợp lệ: {name}')
                target = COLAB_ROOT.joinpath(*path.parts)
                if name.endswith('/'):
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with package.open(item) as source, target.open('wb') as output:
                        shutil.copyfileobj(source, output)
    REPO_ROOT = COLAB_ROOT
else:
    current = Path.cwd().resolve()
    REPO_ROOT = next((p for p in (current, *current.parents) if (p / 'scripts' / 'split_data.py').is_file()), None)
    if REPO_ROOT is None:
        raise RuntimeError('Không tìm thấy repo root. Trên Colab hãy mở notebook bằng Google Colab và upload ZIP khi được hỏi.')
os.chdir(REPO_ROOT)
OUT_DIR = REPO_ROOT / f'submission_{STUDENT_ID}'
CODE_DIR = REPO_ROOT / 'code'
if not (CODE_DIR / 'data.py').is_file():
    raise FileNotFoundError(f'Không thấy {CODE_DIR / "data.py"}; kiểm tra nội dung ZIP trên Drive.')
sys.path.insert(0, str(CODE_DIR))
sys.modules.pop('data', None)
importlib.invalidate_caches()
from data import prepare_data
from plots import plot_compare
from lab_workflow import (initialize_output, run_health_checks, run_baseline_suite,
                          run_topic_experiments, finalize_predictions,
                          refresh_eval_artifacts, print_next_evaluation_command)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
OUT_DIR, FIGURES_DIR, RESULTS_DIR = initialize_output(REPO_ROOT, OUT_DIR)
print(f'PyTorch {torch.__version__} | device={device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(device)}')
print(f'Repo: {REPO_ROOT}')
print(f'Submission: {OUT_DIR}')
print('Notebook nộp bài:', OUT_DIR / 'code' / 'lab.ipynb')

Mounted at /content/drive
Chọn file track4_day1_colab_2A202602777_v2.zip từ máy của bạn.


Saving track4_day1_colab_2A202602777_v2.zip to track4_day1_colab_2A202602777_v2.zip
PyTorch 2.11.0+cu130 | device=cuda
GPU: Tesla T4
Repo: /content/drive/MyDrive/Track4-Day1-2A202602777-v2
Submission: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777
Notebook nộp bài: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/code/lab.ipynb


## Part 0 — Dữ liệu
Đọc split cố định đã tạo từ metadata; tách validation phân tầng 20% chỉ từ train. Thống kê chuẩn hóa chỉ được fit trên phần train còn lại, rồi dùng nguyên thống kê đó cho validation và eval. Eval không tham gia chọn mô hình.

In [ ]:
data = prepare_data(
    device=device, val_fraction=0.2, seed=42,
    processed_dir=REPO_ROOT / 'data' / 'processed',
)
print('Numeric train means:', np.round(data['numeric_mean'], 5))
print('Numeric train stds: ', np.round(data['numeric_std'], 5))
print('Tensor device:', data['X_tr'].device)

train=(371847, 54), val=(92962, 54), eval=(116203, 54), majority_val_acc=0.4876 (class 1)
Numeric train means: [2959.3762   155.78143   14.10812  269.33212   46.42648 2349.8682
  212.15347  223.30785  142.50598 1978.8688 ]
Numeric train stds:  [ 280.1314   111.9549     7.48995  212.23006   58.2381  1557.9949
   26.75816   19.76976   38.27201 1322.636  ]
Tensor device: cuda:0


## Part 1 — Model và kiểm tra sức khỏe
Kiểm tra M-base 54 → 256 → 128 → 7, 47,879 tham số, logits thô, loss ban đầu, gradient và khả năng quá khớp 20 mẫu. Các kết quả đo sẽ được in và đường cong overfit được lưu vào figures/.

In [ ]:
health = run_health_checks(data, FIGURES_DIR)
health

PyTorch 2.11.0+cu130 | device=cuda:0
GPU: Tesla T4
train / val / eval counts: (371847, 92962, 116203)
standardized train mean max abs=3.61437e-07
standardized train std range=1..1
M-base parameters=47879, logits=(8, 7), step0_loss=2.377572 (ln 7=1.945910)
gradient norms: {'net.0.weight': 0.75861, 'net.0.bias': 0.39669, 'net.2.weight': 2.67626, 'net.2.bias': 0.50746, 'net.4.weight': 2.27574, 'net.4.bias': 0.60855}
20-sample overfit: final_loss=0.000081, accuracy=1.000


{'step0_loss': 2.3775719671999713,
 'gradient_norms': {'net.0.weight': 0.7586148381233215,
  'net.0.bias': 0.39668622612953186,
  'net.2.weight': 2.6762616634368896,
  'net.2.bias': 0.5074593424797058,
  'net.4.weight': 2.2757368087768555,
  'net.4.bias': 0.6085489392280579},
 'overfit_loss': 8.085052104433998e-05,
 'overfit_accuracy': 1.0}

**Nhận xét Part 1 — kết quả đo**

`M-base` có 47,879 tham số và cho logits shape `(8, 7)`. Loss bước 0 là `2.3776`, cao hơn `ln(7) = 1.9459` khoảng `0.4317`; đây là giá trị khởi tạo đo được, chưa phải lỗi vì logits ban đầu chưa được hiệu chỉnh. Cả sáu tensor tham số đều có gradient norm khác 0 (nhỏ nhất khoảng `0.3967`). Thử overfit 20 mẫu đạt loss `0.000081` và accuracy `1.000`, cho thấy vòng forward/backward/update có thể học trên tập nhỏ. Đây chưa nói lên khả năng tổng quát hóa.

## Part 2 — Pipeline và baseline
Dò learning rate trên validation cho SGD, SGD+momentum, Adam và AdamW; sau đó chọn learning rate tốt nhất của SGD+momentum để chạy baseline với 3 seed. Dò optimizer ngắn 5 epoch, baseline 20 epoch. Eval không được dùng ở bước chọn.

In [ ]:
baseline_suite = run_baseline_suite(data, FIGURES_DIR, RESULTS_DIR, probe_epochs=5, seeds=(1, 2, 3))
optimizer_probes = baseline_suite['optimizer_probes']
baseline_results = baseline_suite['baseline_results']
print('Selected baseline config:', baseline_suite['baseline_cfg'])
print('Baseline seed noise:', baseline_suite['noise'])


Prediction before opt-sgd-lr0p01: Without momentum, SGD may need a more careful learning-rate choice and may progress less smoothly.
opt-sgd-lr0p01 epoch 01/5: train_loss=0.7558 val_loss=0.7605 val_acc=0.7008 val_macro_f1=0.3159
opt-sgd-lr0p01 epoch 02/5: train_loss=0.6774 val_loss=0.6821 val_acc=0.7194 val_macro_f1=0.3883
opt-sgd-lr0p01 epoch 03/5: train_loss=0.6437 val_loss=0.6482 val_acc=0.7305 val_macro_f1=0.4198
opt-sgd-lr0p01 epoch 04/5: train_loss=0.6245 val_loss=0.6290 val_acc=0.7384 val_macro_f1=0.4494
opt-sgd-lr0p01 epoch 05/5: train_loss=0.6100 val_loss=0.6143 val_acc=0.7432 val_macro_f1=0.4696
After opt-sgd-lr0p01: best epoch=5, val_acc=0.7432, val_macro_f1=0.4696, diverged=False

Prediction before opt-sgd-lr0p03: Without momentum, SGD may need a more careful learning-rate choice and may progress less smoothly.
opt-sgd-lr0p03 epoch 01/5: train_loss=0.6456 val_loss=0.6497 val_acc=0.7304 val_macro_f1=0.4069
opt-sgd-lr0p03 epoch 02/5: train_loss=0.6000 val_loss=0.6043 val_acc

**Nhận xét baseline — kết quả đo**

Mốc validation đoán lớp đa số (class 1) là accuracy `0.4876`; baseline đạt accuracy trung bình `0.9082 ± 0.0024` và macro-F1 `0.8535 ± 0.0126` qua ba seed (`base-s1`, `base-s2`, `base-s3`). Ngưỡng tham khảo `2σ` của macro-F1 là `0.0251`. Ở `base-s1`, tại epoch tốt nhất 18, train loss là `0.2184` và validation loss là `0.2379` (gap `0.0195`); hai loss vẫn khá gần nhau, chưa có dấu hiệu train tốt lên trong khi validation tách xa rõ rệt. Vì vậy dropout chưa có bằng chứng cần thiết chỉ từ baseline này. Các seed cho thấy kết quả có dao động đáng kể, nên chênh lệch nhỏ hơn `0.0251` cần được diễn giải thận trọng.

## Part 3 — Thí nghiệm
Mỗi cấu hình lưu dự đoán trước trong log, chỉ đổi yếu tố ghi trong mô tả, sau đó lưu JSON và hình theo exp_id. Các nhóm gồm loss, batch size, dropout, clipping, mixed precision (nếu phần cứng CUDA hỗ trợ) và khởi tạo.

### Giả thuyết trước các lần chạy

Trước từng lần train, code đã in một dòng `Prediction before ...` trong output. Các giả thuyết cơ chế đó là mốc để đối chiếu ở phần sau: cần dò learning rate; momentum/adaptive optimizer có thể giúp tiến triển nhanh hơn; MSE có thể cập nhật kém hữu ích hơn CE; batch lớn giảm nhiễu gradient nhưng có ít bước cập nhật hơn mỗi epoch; dropout chỉ có lợi nếu baseline quá khớp; clipping có thể hạn chế cập nhật lớn khi learning rate cao; mixed precision cần được đánh giá bằng cả tốc độ, bộ nhớ và độ ổn định; khởi tạo toàn số 0 giữ đối xứng giữa neuron. Phần dưới đây là nhận xét sau khi đã xem kết quả chạy.

In [ ]:
topic_suite = run_topic_experiments(data, baseline_suite, FIGURES_DIR, RESULTS_DIR)
topic_results = topic_suite['results']
clip_norm = topic_suite['clip_norm']
print(f'Clipping threshold from baseline gradient norms: {clip_norm:.6g}')
print(f'Saved {len(topic_results)} topic result records and comparison plots to {FIGURES_DIR}')


Prediction before loss-mse: MSE may make less useful updates on confidently wrong classes; compare validation metrics and convergence, not raw loss values.
loss-mse epoch 01/20: train_loss=0.0483 val_loss=0.0485 val_acc=0.7645 val_macro_f1=0.4526
loss-mse epoch 02/20: train_loss=0.0444 val_loss=0.0444 val_acc=0.7885 val_macro_f1=0.5179
loss-mse epoch 03/20: train_loss=0.0416 val_loss=0.0419 val_acc=0.8006 val_macro_f1=0.5270
loss-mse epoch 04/20: train_loss=0.0402 val_loss=0.0403 val_acc=0.8134 val_macro_f1=0.5785
loss-mse epoch 05/20: train_loss=0.0382 val_loss=0.0385 val_acc=0.8198 val_macro_f1=0.6164
loss-mse epoch 06/20: train_loss=0.0367 val_loss=0.0372 val_acc=0.8282 val_macro_f1=0.6302
loss-mse epoch 07/20: train_loss=0.0355 val_loss=0.0360 val_acc=0.8345 val_macro_f1=0.6578
loss-mse epoch 08/20: train_loss=0.0345 val_loss=0.0350 val_acc=0.8411 val_macro_f1=0.6758
loss-mse epoch 09/20: train_loss=0.0339 val_loss=0.0343 val_acc=0.8442 val_macro_f1=0.6635
loss-mse epoch 10/20: tr

**Đối chiếu sau khi chạy — validation macro-F1**

Mốc nền là trung bình baseline `0.8535`, độ lệch chuẩn `0.0126`, với `2σ = 0.0251`. Dùng ngưỡng này như tham chiếu cho dao động seed, không xem nó là kiểm định thống kê cho từng thí nghiệm đơn lẻ.

- **Optimizer:** các sweep đều chạy 5 epoch, nên so sánh công bằng trong chính nhóm sweep, không trực tiếp với baseline 20 epoch. Learning rate tốt nhất trong các giá trị đã thử là `0.1` cho SGD (`0.6179`) và SGD+momentum (`0.7825`), `0.003` cho Adam (`0.8111`) và AdamW (`0.8111`). Momentum cải thiện `0.1646` so với SGD tại learning rate tốt nhất của mỗi nhóm; Adam/AdamW cao nhất và bằng nhau trong sweep này. Kết quả cho thấy learning rate ảnh hưởng mạnh, nên không kết luận optimizer chỉ từ một learning rate.
- **Loss:** `loss-mse` đạt macro-F1 `0.7277`, thấp hơn baseline trung bình `0.1258`, lớn hơn mốc `2σ`. MSE loss và cross-entropy có thang đo khác nhau; so sánh ở đây dựa trên macro-F1 và đường hội tụ, không so trị số loss trực tiếp.
- **Batch:** `batch-2048` đạt `0.7970`, thấp hơn baseline trung bình `0.0565`. Cùng 20 epoch nhưng batch lớn hơn tạo ít lần cập nhật tham số hơn, phù hợp với việc mô hình tiến triển chậm hơn trong ngân sách epoch này.
- **Dropout:** `dropout-0p3` đạt `0.7875`, thấp hơn baseline trung bình `0.0660`. Ở epoch 20, train/validation loss lần lượt là `0.3109`/`0.3151`, khoảng cách nhỏ nhưng cả hai còn cao hơn baseline tại epoch tốt nhất (`0.2184`/`0.2379`). Kết quả này phù hợp với việc dropout làm chậm học; khoảng cách nhỏ hơn không đồng nghĩa macro-F1 tốt hơn.
- **Clipping:** với learning rate cao `1.0` và ngưỡng clipping `0.737062` lấy từ percentile 95 của norm gradient baseline, macro-F1 tăng từ `0.7732` (không clipping) lên `0.8202` (clipping), chênh `0.0470`. Cả hai run đều không bị đánh dấu diverged, nên ở thí nghiệm này clipping cải thiện kết quả nhưng không phải cứu một run đã diverge.
- **Mixed precision:** FP16 đạt `0.8237` nhưng bị đánh dấu `diverged=True`; BF16 đạt `0.8366` và không diverge. Cả hai thấp hơn baseline trung bình. Output huấn luyện không in thời gian/epoch và bộ nhớ cực đại; hai số đo này có trong các cột `time_per_epoch_s` và `peak_mem_MB` của `experiments.xlsx`. Chỉ so sánh tốc độ/bộ nhớ sau khi xem các cột đó; macro-F1 riêng không chứng minh mixed precision nhanh hơn.
- **Khởi tạo:** zero initialization chỉ đạt accuracy `0.4876` và macro-F1 `0.0936`, phù hợp với giả thuyết các neuron bị giữ đối xứng và không học được biểu diễn hữu ích. Normal đạt `0.8449`, Xavier đạt `0.8514`; cả hai cách nhau dưới `2σ` so với baseline trung bình, nên run đơn lẻ này chưa cho thấy khác biệt rõ ràng với He baseline.

Các so sánh cấu hình Part 3 dựa trên validation; không dùng eval để chọn cấu hình.

In [ ]:
comparison_groups = topic_suite['comparison_groups']
for group, group_runs in comparison_groups.items():
    plot_compare(
        group_runs, 'val_loss', FIGURES_DIR / f'compare_{group}_loss.png',
        title=f'Validation loss: {group}',
    )
print('Comparison figures:', sorted(p.name for p in FIGURES_DIR.glob('compare_*.png')))

Comparison figures: ['compare_amp.png', 'compare_amp_loss.png', 'compare_clipping.png', 'compare_clipping_loss.png', 'compare_dropout.png', 'compare_dropout_loss.png', 'compare_hparam.png', 'compare_hparam_loss.png', 'compare_init.png', 'compare_init_loss.png', 'compare_loss.png', 'compare_loss_loss.png', 'compare_optimizer.png', 'compare_optimizer_loss.png']


## Part 4 — Eval cuối, bảng và hồ sơ nộp
Chọn cấu hình chỉ bằng validation, huấn luyện lại seed 1, rồi tạo dự đoán eval cho baseline và cấu hình cuối. Bước này không tự chạy scripts/evaluate.py; notebook in đúng lệnh kế tiếp để bạn tự chạy trong một ô code Colab.

In [21]:
from pathlib import Path
from hashlib import sha256
import importlib
import os

expected_hash = "20f7ffc6b0fa1e642db5faaa1aa6c57b01f3a28f70e36c6caf6618f9c1919b6e"
candidate = next(
    (p for p in Path("/content").glob("results_table*.py")
     if sha256(p.read_bytes()).hexdigest() == expected_hash),
    None,
)

if candidate is None:
    from google.colab import files
    previous_dir = Path.cwd()
    os.chdir("/content")
    try:
        files.upload()
    finally:
        os.chdir(previous_dir)
    candidate = next(
        (p for p in Path("/content").glob("results_table*.py")
         if sha256(p.read_bytes()).hexdigest() == expected_hash),
        None,
    )

if candidate is None:
    raise ValueError("Không tìm thấy bản results_table.py đã sửa.")

payload = candidate.read_bytes()
old_hash = "d8e5bff41b56a6fe75acd6dd38eca0b5b13d2ef9b93d697a85d2cd01f9e1784d"
targets = [REPO_ROOT / "code/results_table.py", OUT_DIR / "code/results_table.py"]

for target in targets:
    if not target.is_file() or sha256(target.read_bytes()).hexdigest() not in {old_hash, expected_hash}:
        raise ValueError(f"File trên Drive khác bản dự kiến: {target}")
for target in targets:
    target.write_bytes(payload)

import results_table, lab_workflow
importlib.reload(results_table)
importlib.reload(lab_workflow)

all_results = results_table.load_results(RESULTS_DIR)
final_result = next(r for r in all_results if r["cfg"]["exp_id"] == "final-s1")
final_artifacts = {"all_results": all_results, "final_result": final_result}
lab_workflow.refresh_eval_artifacts(all_results, baseline_suite, final_result, OUT_DIR, REPO_ROOT)
lab_workflow.print_next_evaluation_command(OUT_DIR)

Workbook refreshed: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/experiments.xlsx
Next Colab code cell command (run from repo root): !python scripts/evaluate.py --pred submission_2A202602777/predictions_baseline.csv --out submission_2A202602777/eval_result_baseline.json


In [22]:
!python scripts/evaluate.py --pred submission_2A202602777/predictions_baseline.csv --out submission_2A202602777/eval_result_baseline.json

n_eval = 116203
accuracy = 0.9033
macro_f1 = 0.8410   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9016  0.9029  0.9022
  1    56661     0.9093  0.9304  0.9197
  2     7151     0.8972  0.8839  0.8905
  3      549     0.7618  0.7923  0.7768
  4     1899     0.8161  0.6403  0.7176
  5     3473     0.8489  0.7328  0.7866
  6     4102     0.9436  0.8484  0.8935

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  38255   3897     0    0    23     5   188
1   3557  52719    84    1   191    89    20
2      4    359  6321   88    48   331     0
3      0      0    92  435     0    22     0
4     64    600    13    0  1216     6     0
5      9    325   535   47    12  2545     0
6    543     79     0    0     0     0  3480

đã ghi submission_2A202602777/eval_result_baseline.json


In [23]:
!python scripts/evaluate.py --pred submission_2A202602777/predictions_eval.csv --out submission_2A202602777/eval_result.json

n_eval = 116203
accuracy = 0.9033
macro_f1 = 0.8410   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9016  0.9029  0.9022
  1    56661     0.9093  0.9304  0.9197
  2     7151     0.8972  0.8839  0.8905
  3      549     0.7618  0.7923  0.7768
  4     1899     0.8161  0.6403  0.7176
  5     3473     0.8489  0.7328  0.7866
  6     4102     0.9436  0.8484  0.8935

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  38255   3897     0    0    23     5   188
1   3557  52719    84    1   191    89    20
2      4    359  6321   88    48   331     0
3      0      0    92  435     0    22     0
4     64    600    13    0  1216     6     0
5      9    325   535   47    12  2545     0
6    543     79     0    0     0     0  3480

đã ghi submission_2A202602777/eval_result.json


In [24]:
lab_workflow.refresh_eval_artifacts(
    all_results, baseline_suite, final_artifacts["final_result"], OUT_DIR, REPO_ROOT
)

Workbook refreshed: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/experiments.xlsx


In [26]:
from pathlib import Path
import csv, json
from openpyxl import load_workbook

root = Path(OUT_DIR)
required = [
    "REPORT.md", "experiments.xlsx", "predictions_eval.csv",
    "eval_result.json", "code/lab.ipynb",
]
print("Required files:", {name: (root / name).is_file() for name in required})

with (root / "predictions_eval.csv").open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
valid = (
    reader.fieldnames == ["row_id", "pred"]
    and len(rows) == 116203
    and len({r["row_id"] for r in rows}) == 116203
    and all(r["pred"].isdigit() and 0 <= int(r["pred"]) <= 6 for r in rows)
)
print("Predictions valid:", valid, "| rows:", len(rows))

result = json.loads((root / "eval_result.json").read_text(encoding="utf-8"))
print("Final eval:", result["n_eval"], "rows | accuracy:", result["accuracy"], "| macro-F1:", result["macro_f1"])

wb = load_workbook(root / "experiments.xlsx", data_only=False)
ws = wb["Experiments"]
ids = [ws.cell(r, 1).value for r in range(2, ws.max_row + 1) if ws.cell(r, 1).value]
missing = [exp_id for exp_id in ids if not (root / "figures" / f"{exp_id}.png").is_file()]
print("Workbook sheets:", wb.sheetnames, "| experiments:", len(ids), "| missing experiment figures:", missing)
print("REPORT contains final score:", f'{result["macro_f1"]:.4f}' in (root / "REPORT.md").read_text(encoding="utf-8"))

Required files: {'REPORT.md': True, 'experiments.xlsx': True, 'predictions_eval.csv': True, 'eval_result.json': True, 'code/lab.ipynb': True}
Predictions valid: True | rows: 116203
Final eval: 116203 rows | accuracy: 0.9033415660525116 | macro-F1: 0.8409922038067058
Workbook sheets: ['Legend', 'Experiments', 'Seeds', 'Summary'] | experiments: 26 | missing experiment figures: []
REPORT contains final score: True


In [27]:
final_artifacts = finalize_predictions(
    data, baseline_suite, topic_suite, FIGURES_DIR, RESULTS_DIR, OUT_DIR, REPO_ROOT
)
all_results = final_artifacts['all_results']
print('Selected by validation from:', final_artifacts['selected_from'])

Selected by validation: base-s3 (val macro-F1=0.8614)

Prediction before final-s1: Selected on validation from base-s3; no eval scores used for selection.
final-s1 epoch 01/20: train_loss=0.4627 val_loss=0.4675 val_acc=0.7997 val_macro_f1=0.6243
final-s1 epoch 02/20: train_loss=0.3950 val_loss=0.3996 val_acc=0.8300 val_macro_f1=0.7038
final-s1 epoch 03/20: train_loss=0.3792 val_loss=0.3862 val_acc=0.8357 val_macro_f1=0.7256
final-s1 epoch 04/20: train_loss=0.3721 val_loss=0.3800 val_acc=0.8455 val_macro_f1=0.7412
final-s1 epoch 05/20: train_loss=0.3109 val_loss=0.3204 val_acc=0.8671 val_macro_f1=0.7825
final-s1 epoch 06/20: train_loss=0.3042 val_loss=0.3151 val_acc=0.8707 val_macro_f1=0.7949
final-s1 epoch 07/20: train_loss=0.2937 val_loss=0.3047 val_acc=0.8745 val_macro_f1=0.7924
final-s1 epoch 08/20: train_loss=0.2806 val_loss=0.2936 val_acc=0.8780 val_macro_f1=0.7961
final-s1 epoch 09/20: train_loss=0.2925 val_loss=0.3068 val_acc=0.8772 val_macro_f1=0.8026
final-s1 epoch 10/20: trai

In [28]:
for label, filename in [('baseline', 'eval_result_baseline.json'), ('final', 'eval_result.json')]:
    path = OUT_DIR / filename
    if not path.is_file():
        print(f'{label}: chờ bạn chạy evaluator trong một ô code Colab ({filename})')
        continue
    result = json.loads(path.read_text(encoding='utf-8'))
    n_eval, accuracy, macro_f1 = result['n_eval'], result['accuracy'], result['macro_f1']
    print('{}: n={} accuracy={:.4f} macro-F1={:.4f}'.format(label, n_eval, accuracy, macro_f1))
    if label == 'final':
        print('Per-class metrics:')
        for item in result['per_class']:
            print(item)
        print('Confusion matrix (rows=true, columns=predicted):')
        print(np.asarray(result['confusion_matrix']))
print_next_evaluation_command(OUT_DIR)

baseline: n=116203 accuracy=0.9033 macro-F1=0.8410
final: n=116203 accuracy=0.9033 macro-F1=0.8410
Per-class metrics:
{'cls': 0, 'support': 42368, 'precision': 0.9015601432880844, 'recall': 0.9029220166163142, 'f1': 0.9022405660377357}
{'cls': 1, 'support': 56661, 'precision': 0.9092774970247849, 'recall': 0.9304283369513422, 'f1': 0.9197313328681088}
{'cls': 2, 'support': 7151, 'precision': 0.8972320794889993, 'recall': 0.8839323171584393, 'f1': 0.8905325443786982}
{'cls': 3, 'support': 549, 'precision': 0.7618213660245184, 'recall': 0.7923497267759563, 'f1': 0.7767857142857144}
{'cls': 4, 'support': 1899, 'precision': 0.8161073825503355, 'recall': 0.640337019483939, 'f1': 0.7176158158748894}
{'cls': 5, 'support': 3473, 'precision': 0.8488992661774516, 'recall': 0.7327958537287648, 'f1': 0.7865863081440272}
{'cls': 6, 'support': 4102, 'precision': 0.9436008676789588, 'recall': 0.8483666504144319, 'f1': 0.8934531450577664}
Confusion matrix (rows=true, columns=predicted):
[[38255  3897 

**Phân tích lỗi — kết quả eval**

Lớp khó nhất là class 4: support `1,899`, precision `0.8161`, recall `0.6403`, F1 `0.7176`. Trong ma trận nhầm lẫn, `600` mẫu thật class 4 bị đoán thành class 1 và `64` mẫu thành class 0; mô hình bỏ sót nhiều mẫu lớp này. Support class 4 nhỏ hơn nhiều so với class 1 (`56,661`), nên mất cân bằng nhãn là một yếu tố hợp lý cần xem xét, nhưng ma trận không tự chứng minh nguyên nhân đặc trưng. Class 3 có F1 thấp kế tiếp (`0.7768`, support `549`), thường bị đoán thành class 2 (`92` mẫu).

Baseline và cấu hình cuối cùng cùng đạt eval accuracy `0.9033`, macro-F1 `0.8410` trên `116,203` mẫu. Validation chọn `base-s3` (`0.8614`); `final-s1` được train lại với seed 1 và đạt validation macro-F1 `0.8390`, rồi được chấm eval đúng một lần. Điểm eval cuối bằng baseline trong kết quả này, nên không có cải thiện đo được. Hướng thử tiếp theo là class-weighted cross-entropy hoặc batch sampling cân bằng, rồi so macro-F1 và recall class 4 trên validation; đây là đề xuất cho lần thử sau, chưa phải kết quả đã kiểm chứng.

In [29]:
refresh_eval_artifacts(
    all_results, baseline_suite, final_artifacts['final_result'], OUT_DIR, REPO_ROOT
)
print('Workbook:', OUT_DIR / 'experiments.xlsx')
print('Report:', OUT_DIR / 'REPORT.md')
print('Personal interpretation fields in REPORT.md remain for you to complete from saved figures and measured results.')

Workbook refreshed: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/experiments.xlsx
Workbook: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/experiments.xlsx
Report: /content/drive/MyDrive/Track4-Day1-2A202602777-v2/submission_2A202602777/REPORT.md
Personal interpretation fields in REPORT.md remain for you to complete from saved figures and measured results.
